In [2]:
# =================================================================================================
# CELL 4A-T — TENSILE ONLY
# HIGH-PERFORMANCE + RUNTIME-SAFE 10-MODEL SCREENING
# Persistent Optuna | Resume Support | Safe SVR/MLP | 3-Fold CV | Top-5 Selection
# =================================================================================================

import sys
import subprocess
import importlib.util
import warnings
import json
import gc
from pathlib import Path

warnings.filterwarnings("ignore")


# =================================================================================================
# 1. PACKAGE CHECK
# =================================================================================================

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "optuna": "optuna",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm",
    "catboost": "catboost"
}

for module, package in required.items():

    if importlib.util.find_spec(module) is None:

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package
            ]
        )


# =================================================================================================
# 2. IMPORTS
# =================================================================================================

import numpy as np
import pandas as pd
import optuna

from optuna.trial import TrialState

from sklearn.model_selection import (
    train_test_split,
    KFold
)

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    GradientBoostingRegressor
)

from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor

from xgboost import XGBRegressor

from lightgbm import (
    LGBMRegressor,
    early_stopping as lgb_early_stopping
)

from catboost import CatBoostRegressor

from IPython.display import display


optuna.logging.set_verbosity(
    optuna.logging.WARNING
)


# =================================================================================================
# 3. GLOBAL SETTINGS
# =================================================================================================

SEED = 20260906

TEST_SIZE = 0.20

CV_FOLDS = 3

TOP_K = 5


# Strong screening budget.
# SVR/MLP use runtime-safe limits but remain fully included.
TRIAL_BUDGET = {

    "XGBoost": 18,

    "LightGBM": 18,

    "CatBoost": 18,

    "ExtraTrees": 14,

    "RandomForest": 14,

    "HistGradientBoosting": 14,

    "GradientBoosting": 14,

    "SVR_RBF": 12,

    "MLP": 8,

    "KNN": 10
}


MODEL_NAMES = list(
    TRIAL_BUDGET.keys()
)


np.random.seed(
    SEED
)


pd.set_option(
    "display.max_columns",
    100
)

pd.set_option(
    "display.width",
    220
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.6f}"
)


# =================================================================================================
# 4. PROJECT LOCATION
# =================================================================================================

candidate_projects = [

    Path(
        r"C:\Users\apurb\Downloads\Simulation OP ABS"
    ),

    Path.cwd(),

    Path.home() /
    "Downloads" /
    "Simulation OP ABS"

]


PROJECT_ROOT = None


for root in candidate_projects:

    manifest_path = (
        root /
        "Q1_ML_Output" /
        "Cell3_Response_Simulation" /
        "Cell3_Manifest.json"
    )

    if manifest_path.exists():

        PROJECT_ROOT = root.resolve()

        CELL3_MANIFEST = (
            manifest_path.resolve()
        )

        break


if PROJECT_ROOT is None:

    raise FileNotFoundError(
        "Cell 3 output not found."
    )


with open(
    CELL3_MANIFEST,
    "r",
    encoding="utf-8"
) as f:

    manifest3 = json.load(f)


DATA_FILE = Path(
    manifest3[
        "final_simulation_dataset"
    ]
)


if not DATA_FILE.exists():

    raise FileNotFoundError(
        f"Cell 3 dataset not found:\n{DATA_FILE}"
    )


# =================================================================================================
# 5. LOAD DATA
# =================================================================================================

data = pd.read_csv(
    DATA_FILE
)


INPUT_COLS = [

    "GF_pct",

    "Infill_pct",

    "Layer_mm",

    "Nozzle_C",

    "Bed_C",

    "Speed_mm_s"
]


TARGET = manifest3.get(

    "ml_target_tensile",

    "Virtual_Tensile_Mean_MPa"
)


required_cols = (
    INPUT_COLS +
    [TARGET]
)


missing_cols = [

    c
    for c in required_cols
    if c not in data.columns

]


if missing_cols:

    raise ValueError(
        f"Missing columns: {missing_cols}"
    )


if data[
    required_cols
].isna().any().any():

    raise ValueError(
        "Missing values found in ML data."
    )


# =================================================================================================
# 6. OUTPUT DIRECTORIES
# =================================================================================================

CELL4_ROOT = (

    PROJECT_ROOT /
    "Q1_ML_Output" /
    "Cell4_ML_Benchmarking"

)


OUTPUT_DIR = (

    CELL4_ROOT /
    "Cell4A_Tensile"

)


TABLE_DIR = (
    OUTPUT_DIR /
    "Tables"
)


PARAM_DIR = (
    OUTPUT_DIR /
    "Best_Params"
)


OPTUNA_DIR = (
    OUTPUT_DIR /
    "Optuna"
)


OOF_DIR = (
    OUTPUT_DIR /
    "OOF_Predictions"
)


VALIDATION_DIR = (
    OUTPUT_DIR /
    "Validation"
)


for folder in [

    OUTPUT_DIR,

    TABLE_DIR,

    PARAM_DIR,

    OPTUNA_DIR,

    OOF_DIR,

    VALIDATION_DIR

]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# Same DB from previous Cell 4A-T.
# Therefore completed XGBoost -> GradientBoosting studies can be reused.
OPTUNA_DB = (

    OUTPUT_DIR /
    "Tensile_Optuna.sqlite3"

)


STORAGE_URL = (
    f"sqlite:///{OPTUNA_DB.as_posix()}"
)


# =================================================================================================
# 7. FIXED DEVELOPMENT / TEST SPLIT
# =================================================================================================

X_ALL = (

    data[
        INPUT_COLS
    ]
    .to_numpy(
        dtype=float
    )

)


y_ALL = (

    data[
        TARGET
    ]
    .to_numpy(
        dtype=float
    )

)


ALL_INDICES = np.arange(
    len(data)
)


DEV_IDX, TEST_IDX = train_test_split(

    ALL_INDICES,

    test_size=TEST_SIZE,

    shuffle=True,

    random_state=SEED
)


DEV_IDX = np.sort(
    DEV_IDX
)


TEST_IDX = np.sort(
    TEST_IDX
)


X_DEV = X_ALL[
    DEV_IDX
]


y_DEV = y_ALL[
    DEV_IDX
]


split_df = pd.DataFrame({

    "Row_Index":
        ALL_INDICES,

    "Subset":
        np.where(

            np.isin(
                ALL_INDICES,
                DEV_IDX
            ),

            "Development",

            "Independent_Test"

        )

})


if (
    "Simulation_Run_ID"
    in
    data.columns
):

    split_df.insert(

        1,

        "Simulation_Run_ID",

        data[
            "Simulation_Run_ID"
        ].astype(str)

    )


split_df.to_csv(

    VALIDATION_DIR /
    "Fixed_Train_Test_Split.csv",

    index=False
)


# =================================================================================================
# 8. FIXED 3-FOLD CV
# =================================================================================================

cv = KFold(

    n_splits=CV_FOLDS,

    shuffle=True,

    random_state=SEED

)


CV_SPLITS = list(
    cv.split(
        X_DEV
    )
)


# =================================================================================================
# 9. METRICS
# =================================================================================================

def calculate_metrics(
    y_true,
    y_pred
):

    rmse = np.sqrt(

        mean_squared_error(
            y_true,
            y_pred
        )

    )


    return {

        "R2":
            float(
                r2_score(
                    y_true,
                    y_pred
                )
            ),

        "RMSE":
            float(
                rmse
            ),

        "MAE":
            float(
                mean_absolute_error(
                    y_true,
                    y_pred
                )
            )
    }


# =================================================================================================
# 10. STUDY VERSION
#
# v3 = reuse old completed studies
# v4_safe = NEW safe SVR / MLP studies
# =================================================================================================

def study_version(
    model_name
):

    if model_name in [
        "SVR_RBF",
        "MLP"
    ]:

        return "v4_safe"

    return "v3"


# =================================================================================================
# 11. SEARCH SPACES
# =================================================================================================

def suggest_params(
    trial,
    name
):

    # ---------------------------------------------------------------------------------------------
    # XGBOOST
    # ---------------------------------------------------------------------------------------------

    if name == "XGBoost":

        return {

            "n_estimators":
                trial.suggest_int(
                    "n_estimators",
                    300,
                    1100
                ),

            "max_depth":
                trial.suggest_int(
                    "max_depth",
                    3,
                    10
                ),

            "learning_rate":
                trial.suggest_float(
                    "learning_rate",
                    0.005,
                    0.18,
                    log=True
                ),

            "min_child_weight":
                trial.suggest_float(
                    "min_child_weight",
                    0.1,
                    20,
                    log=True
                ),

            "subsample":
                trial.suggest_float(
                    "subsample",
                    0.65,
                    1.00
                ),

            "colsample_bytree":
                trial.suggest_float(
                    "colsample_bytree",
                    0.65,
                    1.00
                ),

            "reg_alpha":
                trial.suggest_float(
                    "reg_alpha",
                    1e-8,
                    10,
                    log=True
                ),

            "reg_lambda":
                trial.suggest_float(
                    "reg_lambda",
                    1e-4,
                    50,
                    log=True
                )
        }


    # ---------------------------------------------------------------------------------------------
    # LIGHTGBM
    # ---------------------------------------------------------------------------------------------

    if name == "LightGBM":

        return {

            "n_estimators":
                trial.suggest_int(
                    "n_estimators",
                    300,
                    1100
                ),

            "learning_rate":
                trial.suggest_float(
                    "learning_rate",
                    0.005,
                    0.18,
                    log=True
                ),

            "num_leaves":
                trial.suggest_int(
                    "num_leaves",
                    15,
                    127
                ),

            "max_depth":
                trial.suggest_int(
                    "max_depth",
                    3,
                    10
                ),

            "min_child_samples":
                trial.suggest_int(
                    "min_child_samples",
                    5,
                    70
                ),

            "subsample":
                trial.suggest_float(
                    "subsample",
                    0.65,
                    1.00
                ),

            "colsample_bytree":
                trial.suggest_float(
                    "colsample_bytree",
                    0.65,
                    1.00
                ),

            "reg_alpha":
                trial.suggest_float(
                    "reg_alpha",
                    1e-8,
                    10,
                    log=True
                ),

            "reg_lambda":
                trial.suggest_float(
                    "reg_lambda",
                    1e-4,
                    50,
                    log=True
                )
        }


    # ---------------------------------------------------------------------------------------------
    # CATBOOST
    # ---------------------------------------------------------------------------------------------

    if name == "CatBoost":

        return {

            "iterations":
                trial.suggest_int(
                    "iterations",
                    300,
                    1100
                ),

            "depth":
                trial.suggest_int(
                    "depth",
                    4,
                    10
                ),

            "learning_rate":
                trial.suggest_float(
                    "learning_rate",
                    0.005,
                    0.18,
                    log=True
                ),

            "l2_leaf_reg":
                trial.suggest_float(
                    "l2_leaf_reg",
                    0.01,
                    100,
                    log=True
                ),

            "random_strength":
                trial.suggest_float(
                    "random_strength",
                    1e-5,
                    10,
                    log=True
                )
        }


    # ---------------------------------------------------------------------------------------------
    # EXTRA TREES
    # ---------------------------------------------------------------------------------------------

    if name == "ExtraTrees":

        return {

            "n_estimators":
                trial.suggest_int(
                    "n_estimators",
                    350,
                    850
                ),

            "max_depth":
                trial.suggest_categorical(
                    "max_depth",
                    [
                        None,
                        12,
                        18,
                        24,
                        32
                    ]
                ),

            "min_samples_split":
                trial.suggest_int(
                    "min_samples_split",
                    2,
                    12
                ),

            "min_samples_leaf":
                trial.suggest_int(
                    "min_samples_leaf",
                    1,
                    6
                ),

            "max_features":
                trial.suggest_float(
                    "max_features",
                    0.50,
                    1.00
                )
        }


    # ---------------------------------------------------------------------------------------------
    # RANDOM FOREST
    # ---------------------------------------------------------------------------------------------

    if name == "RandomForest":

        return {

            "n_estimators":
                trial.suggest_int(
                    "n_estimators",
                    350,
                    850
                ),

            "max_depth":
                trial.suggest_categorical(
                    "max_depth",
                    [
                        None,
                        12,
                        18,
                        24,
                        32
                    ]
                ),

            "min_samples_split":
                trial.suggest_int(
                    "min_samples_split",
                    2,
                    12
                ),

            "min_samples_leaf":
                trial.suggest_int(
                    "min_samples_leaf",
                    1,
                    6
                ),

            "max_features":
                trial.suggest_float(
                    "max_features",
                    0.50,
                    1.00
                )
        }


    # ---------------------------------------------------------------------------------------------
    # HIST GRADIENT BOOSTING
    # ---------------------------------------------------------------------------------------------

    if name == "HistGradientBoosting":

        return {

            "max_iter":
                trial.suggest_int(
                    "max_iter",
                    180,
                    800
                ),

            "learning_rate":
                trial.suggest_float(
                    "learning_rate",
                    0.005,
                    0.18,
                    log=True
                ),

            "max_leaf_nodes":
                trial.suggest_int(
                    "max_leaf_nodes",
                    15,
                    127
                ),

            "max_depth":
                trial.suggest_categorical(
                    "max_depth",
                    [
                        None,
                        4,
                        6,
                        8,
                        10
                    ]
                ),

            "min_samples_leaf":
                trial.suggest_int(
                    "min_samples_leaf",
                    5,
                    60
                ),

            "l2_regularization":
                trial.suggest_float(
                    "l2_regularization",
                    1e-8,
                    10,
                    log=True
                )
        }


    # ---------------------------------------------------------------------------------------------
    # GRADIENT BOOSTING
    # ---------------------------------------------------------------------------------------------

    if name == "GradientBoosting":

        return {

            "n_estimators":
                trial.suggest_int(
                    "n_estimators",
                    180,
                    800
                ),

            "learning_rate":
                trial.suggest_float(
                    "learning_rate",
                    0.005,
                    0.18,
                    log=True
                ),

            "max_depth":
                trial.suggest_int(
                    "max_depth",
                    2,
                    6
                ),

            "min_samples_leaf":
                trial.suggest_int(
                    "min_samples_leaf",
                    1,
                    15
                ),

            "subsample":
                trial.suggest_float(
                    "subsample",
                    0.65,
                    1.00
                )
        }


    # ---------------------------------------------------------------------------------------------
    # SVR — RUNTIME SAFE, STILL HIGH PERFORMANCE
    # ---------------------------------------------------------------------------------------------

    if name == "SVR_RBF":

        return {

            "C":
                trial.suggest_float(
                    "C",
                    0.1,
                    2000,
                    log=True
                ),

            "epsilon":
                trial.suggest_float(
                    "epsilon",
                    1e-4,
                    0.20,
                    log=True
                ),

            "gamma":
                trial.suggest_float(
                    "gamma",
                    1e-4,
                    3.0,
                    log=True
                )
        }


    # ---------------------------------------------------------------------------------------------
    # MLP — EXPRESSIVE BUT RUNTIME SAFE
    # ---------------------------------------------------------------------------------------------

    if name == "MLP":

        return {

            "hidden_layers":
                trial.suggest_categorical(
                    "hidden_layers",
                    [
                        "64-32",
                        "128-64",
                        "128-64-32"
                    ]
                ),

            "activation":
                trial.suggest_categorical(
                    "activation",
                    [
                        "relu",
                        "tanh"
                    ]
                ),

            "alpha":
                trial.suggest_float(
                    "alpha",
                    1e-7,
                    1e-2,
                    log=True
                ),

            "learning_rate_init":
                trial.suggest_float(
                    "learning_rate_init",
                    1e-4,
                    1e-2,
                    log=True
                ),

            "batch_size":
                trial.suggest_categorical(
                    "batch_size",
                    [
                        64,
                        128,
                        256
                    ]
                )
        }


    # ---------------------------------------------------------------------------------------------
    # KNN
    # ---------------------------------------------------------------------------------------------

    if name == "KNN":

        return {

            "n_neighbors":
                trial.suggest_int(
                    "n_neighbors",
                    3,
                    70
                ),

            "weights":
                trial.suggest_categorical(
                    "weights",
                    [
                        "uniform",
                        "distance"
                    ]
                ),

            "p":
                trial.suggest_categorical(
                    "p",
                    [
                        1,
                        2
                    ]
                )
        }


    raise ValueError(
        f"Unknown model: {name}"
    )


# =================================================================================================
# 12. MODEL FACTORY
# =================================================================================================

def build_model(
    name,
    params,
    seed,
    early=True
):

    # ---------------------------------------------------------------------------------------------
    # XGBOOST
    # ---------------------------------------------------------------------------------------------

    if name == "XGBoost":

        extra = {}

        if early:

            extra[
                "early_stopping_rounds"
            ] = 40

        return XGBRegressor(

            **params,

            **extra,

            objective=
                "reg:squarederror",

            eval_metric=
                "rmse",

            tree_method=
                "hist",

            random_state=
                seed,

            n_jobs=
                -1,

            verbosity=
                0
        )


    # ---------------------------------------------------------------------------------------------
    # LIGHTGBM
    # ---------------------------------------------------------------------------------------------

    if name == "LightGBM":

        return LGBMRegressor(

            **params,

            objective=
                "regression",

            random_state=
                seed,

            n_jobs=
                -1,

            subsample_freq=
                1,

            verbosity=
                -1
        )


    # ---------------------------------------------------------------------------------------------
    # CATBOOST
    # ---------------------------------------------------------------------------------------------

    if name == "CatBoost":

        return CatBoostRegressor(

            **params,

            loss_function=
                "RMSE",

            random_seed=
                seed,

            verbose=
                False,

            allow_writing_files=
                False,

            thread_count=
                -1
        )


    # ---------------------------------------------------------------------------------------------
    # EXTRA TREES
    # ---------------------------------------------------------------------------------------------

    if name == "ExtraTrees":

        return ExtraTreesRegressor(

            **params,

            random_state=
                seed,

            n_jobs=
                -1
        )


    # ---------------------------------------------------------------------------------------------
    # RANDOM FOREST
    # ---------------------------------------------------------------------------------------------

    if name == "RandomForest":

        return RandomForestRegressor(

            **params,

            random_state=
                seed,

            n_jobs=
                -1
        )


    # ---------------------------------------------------------------------------------------------
    # HIST GRADIENT BOOSTING
    # ---------------------------------------------------------------------------------------------

    if name == "HistGradientBoosting":

        return HistGradientBoostingRegressor(

            **params,

            early_stopping=
                False,

            random_state=
                seed
        )


    # ---------------------------------------------------------------------------------------------
    # GRADIENT BOOSTING
    # ---------------------------------------------------------------------------------------------

    if name == "GradientBoosting":

        return GradientBoostingRegressor(

            **params,

            random_state=
                seed
        )


    # ---------------------------------------------------------------------------------------------
    # SVR — IMPORTANT RUNTIME LIMITS
    # ---------------------------------------------------------------------------------------------

    if name == "SVR_RBF":

        svr = SVR(

            kernel=
                "rbf",

            C=
                params[
                    "C"
                ],

            epsilon=
                params[
                    "epsilon"
                ],

            gamma=
                params[
                    "gamma"
                ],

            cache_size=
                1024,

            max_iter=
                20000,

            tol=
                1e-3
        )


        pipeline = Pipeline(

            steps=[

                (
                    "X_scaler",
                    StandardScaler()
                ),

                (
                    "SVR",
                    svr
                )
            ]

        )


        return TransformedTargetRegressor(

            regressor=
                pipeline,

            transformer=
                StandardScaler()
        )


    # ---------------------------------------------------------------------------------------------
    # MLP — RUNTIME CONTROL
    # ---------------------------------------------------------------------------------------------

    if name == "MLP":

        layers = tuple(

            int(x)

            for x in

            params[
                "hidden_layers"
            ].split("-")

        )


        mlp = MLPRegressor(

            hidden_layer_sizes=
                layers,

            activation=
                params[
                    "activation"
                ],

            solver=
                "adam",

            alpha=
                params[
                    "alpha"
                ],

            learning_rate_init=
                params[
                    "learning_rate_init"
                ],

            batch_size=
                params[
                    "batch_size"
                ],

            max_iter=
                600,

            early_stopping=
                True,

            validation_fraction=
                0.10,

            n_iter_no_change=
                20,

            tol=
                1e-4,

            random_state=
                seed
        )


        pipeline = Pipeline(

            steps=[

                (
                    "X_scaler",
                    StandardScaler()
                ),

                (
                    "MLP",
                    mlp
                )
            ]

        )


        return TransformedTargetRegressor(

            regressor=
                pipeline,

            transformer=
                StandardScaler()
        )


    # ---------------------------------------------------------------------------------------------
    # KNN
    # ---------------------------------------------------------------------------------------------

    if name == "KNN":

        knn = KNeighborsRegressor(

            n_neighbors=
                params[
                    "n_neighbors"
                ],

            weights=
                params[
                    "weights"
                ],

            p=
                params[
                    "p"
                ],

            n_jobs=
                -1
        )


        pipeline = Pipeline(

            steps=[

                (
                    "X_scaler",
                    StandardScaler()
                ),

                (
                    "KNN",
                    knn
                )
            ]

        )


        return TransformedTargetRegressor(

            regressor=
                pipeline,

            transformer=
                StandardScaler()
        )


    raise ValueError(
        f"Unknown model: {name}"
    )


# =================================================================================================
# 13. FIT MODEL
# =================================================================================================

def fit_model(
    name,
    model,
    X_train,
    y_train,
    X_valid,
    y_valid
):

    if name == "XGBoost":

        model.fit(

            X_train,

            y_train,

            eval_set=[
                (
                    X_valid,
                    y_valid
                )
            ],

            verbose=False
        )


    elif name == "LightGBM":

        model.fit(

            X_train,

            y_train,

            eval_set=[
                (
                    X_valid,
                    y_valid
                )
            ],

            callbacks=[

                lgb_early_stopping(
                    40,
                    verbose=False
                )

            ]
        )


    elif name == "CatBoost":

        model.fit(

            X_train,

            y_train,

            eval_set=(
                X_valid,
                y_valid
            ),

            early_stopping_rounds=
                40,

            use_best_model=
                True,

            verbose=
                False
        )


    else:

        model.fit(
            X_train,
            y_train
        )


    return model


# =================================================================================================
# 14. SCREENING
# =================================================================================================

result_rows = []

best_params_all = {}


print(
    "\n" +
    "=" * 110
)


print(
    "CELL 4A-T — TENSILE STRENGTH HIGH-PERFORMANCE SCREENING"
)


print(
    "=" * 110
)


for model_number, name in enumerate(
    MODEL_NAMES,
    start=1
):

    print(
        f"\n[{model_number:02d}/10] {name}"
    )


    version = study_version(
        name
    )


    study_name = (
        f"Tensile_{name}_Screening_{version}"
    )


    sampler = optuna.samplers.TPESampler(

        seed=
            SEED +
            model_number *
            1000,

        n_startup_trials=
            5,

        multivariate=
            True
    )


    pruner = optuna.pruners.MedianPruner(

        n_startup_trials=
            5,

        n_warmup_steps=
            1
    )


    study = optuna.create_study(

        study_name=
            study_name,

        storage=
            STORAGE_URL,

        direction=
            "minimize",

        sampler=
            sampler,

        pruner=
            pruner,

        load_if_exists=
            True
    )


    completed_before = len([

        trial

        for trial in study.trials

        if (
            trial.state
            ==
            TrialState.COMPLETE
        )

    ])


    target_trials = (
        TRIAL_BUDGET[
            name
        ]
    )


    remaining = max(

        0,

        target_trials -
        completed_before
    )


    print(
        f"   Completed trials: "
        f"{completed_before}/{target_trials}"
    )


    # =============================================================================================
    # OPTUNA
    # =============================================================================================

    if remaining > 0:

        def objective(
            trial
        ):

            params = suggest_params(
                trial,
                name
            )


            fold_scores = []


            for fold, (
                train_idx,
                valid_idx
            ) in enumerate(
                CV_SPLITS,
                start=1
            ):

                model = build_model(

                    name,

                    params,

                    seed=(
                        SEED +
                        trial.number *
                        101 +
                        fold
                    ),

                    early=True
                )


                try:

                    model = fit_model(

                        name,

                        model,

                        X_DEV[
                            train_idx
                        ],

                        y_DEV[
                            train_idx
                        ],

                        X_DEV[
                            valid_idx
                        ],

                        y_DEV[
                            valid_idx
                        ]
                    )


                    prediction = model.predict(

                        X_DEV[
                            valid_idx
                        ]

                    )


                    rmse = float(

                        np.sqrt(

                            mean_squared_error(

                                y_DEV[
                                    valid_idx
                                ],

                                prediction

                            )

                        )

                    )


                    if not np.isfinite(
                        rmse
                    ):

                        raise ValueError(
                            "Non-finite RMSE."
                        )


                    fold_scores.append(
                        rmse
                    )


                    trial.report(

                        float(
                            np.mean(
                                fold_scores
                            )
                        ),

                        step=fold
                    )


                    if (
                        fold >= 2
                        and
                        trial.should_prune()
                    ):

                        raise optuna.TrialPruned()


                finally:

                    del model

                    gc.collect()


            return float(

                np.mean(
                    fold_scores
                )

            )


        study.optimize(

            objective,

            n_trials=
                remaining,

            n_jobs=
                1,

            show_progress_bar=
                False,

            gc_after_trial=
                True
        )


    # =============================================================================================
    # SUCCESS CHECK
    # =============================================================================================

    completed_trials = [

        trial

        for trial in study.trials

        if (
            trial.state
            ==
            TrialState.COMPLETE
        )

    ]


    if len(
        completed_trials
    ) == 0:

        raise RuntimeError(
            f"No successful Optuna trial for {name}."
        )


    best_params = (
        study.best_params
    )


    best_params_all[
        name
    ] = best_params


    # =============================================================================================
    # SAVE OPTUNA
    # =============================================================================================

    study.trials_dataframe().to_csv(

        OPTUNA_DIR /
        f"Tensile_{name}_{version}_Optuna_Trials.csv",

        index=False
    )


    with open(

        PARAM_DIR /
        f"Tensile_{name}_{version}_Best_Params.json",

        "w",

        encoding="utf-8"

    ) as f:

        json.dump(

            best_params,

            f,

            indent=4

        )


    # =============================================================================================
    # OOF CACHE
    # =============================================================================================

    OOF_FILE = (

        OOF_DIR /
        f"Tensile_{name}_{version}_OOF.csv"

    )


    use_cached_oof = (

        OOF_FILE.exists()

        and

        completed_before >=
        target_trials

    )


    if use_cached_oof:

        print(
            "   Loading saved OOF predictions..."
        )


        cached = pd.read_csv(
            OOF_FILE
        )


        oof = (
            cached[
                "OOF_Prediction"
            ]
            .to_numpy(
                dtype=float
            )
        )


        fold_rmse = [

            np.nan,
            np.nan,
            np.nan

        ]


    else:

        # =========================================================================================
        # FRESH OOF VALIDATION
        # =========================================================================================

        oof = np.zeros_like(
            y_DEV,
            dtype=float
        )


        fold_rmse = []


        # Two independent MLP seeds are averaged.
        repeats = (

            2

            if
            name == "MLP"

            else
            1

        )


        for fold, (
            train_idx,
            valid_idx
        ) in enumerate(
            CV_SPLITS,
            start=1
        ):

            repeated_predictions = []


            for repeat in range(
                repeats
            ):

                model = build_model(

                    name,

                    best_params,

                    seed=(
                        SEED +
                        50000 +
                        fold *
                        100 +
                        repeat
                    ),

                    early=True
                )


                model = fit_model(

                    name,

                    model,

                    X_DEV[
                        train_idx
                    ],

                    y_DEV[
                        train_idx
                    ],

                    X_DEV[
                        valid_idx
                    ],

                    y_DEV[
                        valid_idx
                    ]

                )


                repeated_predictions.append(

                    model.predict(

                        X_DEV[
                            valid_idx
                        ]

                    )

                )


                del model

                gc.collect()


            prediction = np.mean(

                repeated_predictions,

                axis=0
            )


            oof[
                valid_idx
            ] = prediction


            fold_rmse.append(

                float(

                    np.sqrt(

                        mean_squared_error(

                            y_DEV[
                                valid_idx
                            ],

                            prediction

                        )

                    )

                )

            )


        oof_df = pd.DataFrame({

            "Development_Row_Index":
                DEV_IDX,

            "Observed_Virtual_Target":
                y_DEV,

            "OOF_Prediction":
                oof,

            "Residual":
                y_DEV -
                oof

        })


        oof_df.to_csv(

            OOF_FILE,

            index=False
        )


    # =============================================================================================
    # METRICS
    # =============================================================================================

    metric = calculate_metrics(

        y_DEV,

        oof
    )


    if np.all(
        np.isnan(
            fold_rmse
        )
    ):

        fold_mean = np.nan

        fold_sd = np.nan

    else:

        fold_mean = float(
            np.nanmean(
                fold_rmse
            )
        )

        fold_sd = float(
            np.nanstd(
                fold_rmse,
                ddof=1
            )
        )


    result_rows.append({

        "Response":
            "Tensile Strength",

        "Model":
            name,

        "Study_Version":
            version,

        "Target_Optuna_Trials":
            target_trials,

        "Completed_Trials":
            len(
                completed_trials
            ),

        "Best_Optuna_CV_RMSE":
            float(
                study.best_value
            ),

        "OOF_R2":
            metric[
                "R2"
            ],

        "OOF_RMSE":
            metric[
                "RMSE"
            ],

        "OOF_MAE":
            metric[
                "MAE"
            ],

        "Fold_RMSE_Mean":
            fold_mean,

        "Fold_RMSE_SD":
            fold_sd

    })


    print(
        f"   OOF R²   = "
        f"{metric['R2']:.6f}"
    )


    print(
        f"   OOF RMSE = "
        f"{metric['RMSE']:.6f} MPa"
    )


    print(
        f"   OOF MAE  = "
        f"{metric['MAE']:.6f} MPa"
    )


# =================================================================================================
# 15. FINAL LEADERBOARD
# =================================================================================================

leaderboard = (

    pd.DataFrame(
        result_rows
    )

    .sort_values(
        "OOF_RMSE",
        ascending=True
    )

    .reset_index(
        drop=True
    )

)


leaderboard[
    "Screening_Rank"
] = np.arange(

    1,

    len(
        leaderboard
    ) + 1

)


TOP5 = (

    leaderboard[
        "Model"
    ]

    .head(
        TOP_K
    )

    .tolist()

)


LEADERBOARD_FILE = (

    TABLE_DIR /
    "Tensile_10_Model_RuntimeSafe_Leaderboard.csv"

)


leaderboard.to_csv(

    LEADERBOARD_FILE,

    index=False

)


# =================================================================================================
# 16. MANIFEST
# =================================================================================================

manifest4a_tensile = {

    "project_root":
        str(
            PROJECT_ROOT
        ),

    "data_file":
        str(
            DATA_FILE
        ),

    "output_dir":
        str(
            OUTPUT_DIR
        ),

    "input_cols":
        INPUT_COLS,

    "target":
        TARGET,

    "dev_indices":
        DEV_IDX.tolist(),

    "test_indices":
        TEST_IDX.tolist(),

    "cv_folds":
        CV_FOLDS,

    "top5":
        TOP5,

    "best_params":
        best_params_all,

    "leaderboard":
        str(
            LEADERBOARD_FILE
        ),

    "optuna_database":
        str(
            OPTUNA_DB
        ),

    "svr_runtime_limit":
        {
            "max_iter":
                20000,

            "tol":
                0.001,

            "cache_size_MB":
                1024
        },

    "mlp_runtime_limit":
        {
            "max_iter":
                600,

            "n_iter_no_change":
                20,

            "tol":
                0.0001
        },

    "seed":
        SEED

}


MANIFEST_FILE = (

    OUTPUT_DIR /
    "Cell4A_Tensile_Manifest.json"

)


with open(

    MANIFEST_FILE,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        manifest4a_tensile,

        f,

        indent=4

    )


# =================================================================================================
# 17. DISPLAY
# =================================================================================================

print(
    "\nTENSILE — FINAL 10-MODEL SCREENING LEADERBOARD"
)


display(

    leaderboard.round(
        6
    )

)


print(
    "\nTOP 5 SELECTED FOR CELL 4B DEEP TUNING"
)


for rank, model_name in enumerate(
    TOP5,
    start=1
):

    print(
        f"{rank}. {model_name}"
    )


print(
    "\n" +
    "=" * 110
)


print(
    "CELL 4A-T — TENSILE SCREENING COMPLETE"
)


print(
    "=" * 110
)


print(
    f"\nTotal virtual observations : "
    f"{len(data)}"
)


print(
    f"Development observations   : "
    f"{len(DEV_IDX)}"
)


print(
    f"Reserved independent test  : "
    f"{len(TEST_IDX)}"
)


print(
    "\nIndependent test set used for model selection: NO"
)


print(
    "\nIMPORTANT:"
)


print(
    "XGBoost-to-GradientBoosting completed Optuna studies are reused when available."
)


print(
    "SVR and MLP use new runtime-safe studies."
)


print(
    "SVR has a hard max_iter limit, so it cannot run indefinitely."
)


print(
    "MLP remains included with scaling, nonlinear hidden layers, early stopping and repeated OOF validation."
)


print(
    f"\nOutputs saved to:\n{OUTPUT_DIR}"
)


print(
    "\nIf interrupted, run this SAME cell again. Saved Optuna studies will be resumed."
)


print(
    "=" * 110
)


CELL 4A-T — TENSILE STRENGTH HIGH-PERFORMANCE SCREENING

[01/10] XGBoost
   Completed trials: 14/18
   OOF R²   = 0.994352
   OOF RMSE = 0.578081 MPa
   OOF MAE  = 0.444688 MPa

[02/10] LightGBM
   Completed trials: 14/18
   OOF R²   = 0.994164
   OOF RMSE = 0.587629 MPa
   OOF MAE  = 0.451448 MPa

[03/10] CatBoost
   Completed trials: 17/18
   OOF R²   = 0.998401
   OOF RMSE = 0.307559 MPa
   OOF MAE  = 0.235517 MPa

[04/10] ExtraTrees
   Completed trials: 13/14
   OOF R²   = 0.926741
   OOF RMSE = 2.081957 MPa
   OOF MAE  = 1.629402 MPa

[05/10] RandomForest
   Completed trials: 11/14
   OOF R²   = 0.928350
   OOF RMSE = 2.058979 MPa
   OOF MAE  = 1.633388 MPa

[06/10] HistGradientBoosting
   Completed trials: 11/14
   OOF R²   = 0.988426
   OOF RMSE = 0.827530 MPa
   OOF MAE  = 0.630700 MPa

[07/10] GradientBoosting
   Completed trials: 14/14
   OOF R²   = 0.995130
   OOF RMSE = 0.536801 MPa
   OOF MAE  = 0.410853 MPa

[08/10] SVR_RBF
   Completed trials: 0/12
   OOF R²   = 0.99737

,Response,Model,Study_Version,Target_Optuna_Trials,Completed_Trials,Best_Optuna_CV_RMSE,OOF_R2,OOF_RMSE,OOF_MAE,Fold_RMSE_Mean,Fold_RMSE_SD,Screening_Rank
0,Tensile Strength,MLP,v4_safe,8,7,0.328672,0.998548,0.293122,0.211543,0.292625,0.020748,1
1,Tensile Strength,CatBoost,v3,18,17,0.305507,0.998401,0.307559,0.235517,0.306818,0.026255,2
2,Tensile Strength,SVR_RBF,v4_safe,12,11,0.393443,0.997377,0.393962,0.274673,0.393443,0.024779,3
3,Tensile Strength,GradientBoosting,v3,14,14,0.532108,0.995130,0.536801,0.410853,0.536316,0.028139,4
4,Tensile Strength,XGBoost,v3,18,18,0.576886,0.994352,0.578081,0.444688,0.577821,0.021322,5
5,Tensile Strength,LightGBM,v3,18,18,0.569155,0.994164,0.587629,0.451448,0.587568,0.010290,6
6,Tensile Strength,HistGradientBoosting,v3,14,13,0.827336,0.988426,0.827530,0.630700,0.827336,0.022416,7
7,Tensile Strength,RandomForest,v3,14,14,2.051431,0.928350,2.058979,1.633388,2.058559,0.051558,8
8,Tensile Strength,ExtraTrees,v3,14,14,2.073914,0.926741,2.081957,1.629402,2.081647,0.045121,9
9,Tensile Strength,KNN,v3,10,9,2.979342,0.849882,2.980294,2.131826,2.979342,0.093895,10



TOP 5 SELECTED FOR CELL 4B DEEP TUNING
1. MLP
2. CatBoost
3. SVR_RBF
4. GradientBoosting
5. XGBoost

CELL 4A-T — TENSILE SCREENING COMPLETE

Total virtual observations : 4000
Development observations   : 3200
Reserved independent test  : 800

Independent test set used for model selection: NO

IMPORTANT:
XGBoost-to-GradientBoosting completed Optuna studies are reused when available.
SVR and MLP use new runtime-safe studies.
SVR has a hard max_iter limit, so it cannot run indefinitely.
MLP remains included with scaling, nonlinear hidden layers, early stopping and repeated OOF validation.

Outputs saved to:
C:\Users\apurb\Downloads\Simulation OP ABS\Q1_ML_Output\Cell4_ML_Benchmarking\Cell4A_Tensile

If interrupted, run this SAME cell again. Saved Optuna studies will be resumed.
